Import all libraries

In [1]:
import pandas as pd
import joblib
import os
import glob
import sys
sys.path.append(os.path.abspath(os.path.join('..','..')))
from config import DATASETS,MODELS_PIPELINES
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report,confusion_matrix


c:\Users\hamen\OneDrive\Desktop\ml_project\Diagnosis_ML_project
Project structure created successfully.


select latest csv file path from Datasets folder

In [2]:
# find latest dataset

def select_latest_file():
    file_paths = glob.glob(os.path.join(DATASETS,"*.csv"))
    if not file_paths:
        return None
    else:
        latest = max(file_paths,key=os.path.getmtime)
        df = pd.read_csv(latest)
        return df

convert csv file data into Dataframe

In [3]:
# load dataframe
try:
    df = select_latest_file()
    print(df.head(5))
except:
    print("no file avilable insdie DATASET folder")

   Patient_ID  Age  Gender            Symptom_1    Symptom_2 Symptom_3  \
0           1   74    Male              Fatigue  Sore throat     Fever   
1           2   66  Female          Sore throat      Fatigue     Cough   
2           3   32    Male            Body ache  Sore throat   Fatigue   
3           4   21  Female  Shortness of breath     Headache     Cough   
4           5   53    Male           Runny nose  Sore throat   Fatigue   

   Heart_Rate_bpm  Body_Temperature_C Blood_Pressure_mmHg  \
0              69                39.4              132/91   
1              95                39.0              174/98   
2              77                36.8              136/60   
3              72                38.9              147/82   
4             100                36.6             109/106   

   Oxygen_Saturation_pct Diagnosis  Severity       Treatment_Plan  
0                     94       Flu  Moderate  Medication and rest  
1                     98   Healthy      Mild      Re

Separate Target features and remaning features

In [4]:
x = df
y = df["Treatment_Plan"]

In [24]:
y.unique()

<StringArray>
['Medication and rest', 'Rest and fluids', 'Hospitalization and medication']
Length: 3, dtype: str

Separate x and y into train and test 

In [5]:
x_train_raw,x_test_raw,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)
x_train_raw, y_test

(      Patient_ID  Age  Gender            Symptom_1            Symptom_2  \
 126          127   46    Male           Runny nose                Cough   
 660          661   79    Male                Cough          Sore throat   
 421          422   30    Male                Cough             Headache   
 1219        1220   28  Female                Fever              Fatigue   
 1721        1722   59    Male             Headache  Shortness of breath   
 ...          ...  ...     ...                  ...                  ...   
 1626        1627   40  Female             Headache                Fever   
 749          750   54  Female           Runny nose            Body ache   
 191          192   62    Male  Shortness of breath              Fatigue   
 702          703   73    Male                Cough           Runny nose   
 1374        1375   77  Female             Headache              Fatigue   
 
                 Symptom_3  Heart_Rate_bpm  Body_Temperature_C  \
 126             Bod

Load Data cleaning pipeline 

In [6]:

file_name = "Disease_data_cleaning_pipeline.pkl"
pipeline_path = os.path.join(MODELS_PIPELINES,file_name)

data_clean_pipeline = joblib.load(pipeline_path)

Clean x_train and x_test for ml models

In [7]:
x_train_pipeline= data_clean_pipeline.transform(x_train_raw)
x_train_clean= pd.DataFrame(x_train_pipeline,columns = data_clean_pipeline.get_feature_names_out())
x_test_pipeline = data_clean_pipeline.transform(x_test_raw)
x_test_clean = pd.DataFrame(x_test_pipeline,columns=data_clean_pipeline.get_feature_names_out())
x_train_clean

,bp__systolic,bp__diastolic,binary__Gender,nominal__Symptom_1_Body ache,nominal__Symptom_1_Cough,nominal__Symptom_1_Fatigue,nominal__Symptom_1_Fever,nominal__Symptom_1_Headache,nominal__Symptom_1_Runny nose,nominal__Symptom_1_Shortness of breath,...,nominal__Symptom_3_Fatigue,nominal__Symptom_3_Fever,nominal__Symptom_3_Headache,nominal__Symptom_3_Runny nose,nominal__Symptom_3_Shortness of breath,nominal__Symptom_3_Sore throat,continue__Age,continue__Heart_Rate_bpm,continue__Body_Temperature_C,continue__Oxygen_Saturation_pct
0,92.0,67.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,46.0,94.0,36.3,96.0
1,178.0,118.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,79.0,119.0,37.4,95.0
2,145.0,82.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,30.0,108.0,37.8,95.0
3,145.0,73.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,28.0,70.0,37.0,96.0
4,160.0,64.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,59.0,61.0,39.9,95.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1595,164.0,85.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,40.0,97.0,37.1,95.0
1596,162.0,118.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,54.0,71.0,38.4,97.0
1597,171.0,108.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,62.0,105.0,35.7,95.0
1598,177.0,62.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,73.0,78.0,37.6,96.0


Logistic Regression

In [8]:
#Define model and fit train data
lr = LogisticRegression(penalty='l2',C=1.0,class_weight="balanced",solver='lbfgs',max_iter=5000)
lr.fit(x_train_clean,y_train)

c:\Users\hamen\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\hamen\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 5000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=5000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-r

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'l2'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multi

predict output using logistic regression

In [9]:
lr_test_pre = lr.predict(x_test_clean)
lr_test_pre

array(['Rest and fluids', 'Medication and rest',
       'Hospitalization and medication', 'Rest and fluids',
       'Rest and fluids', 'Medication and rest',
       'Hospitalization and medication', 'Rest and fluids',
       'Hospitalization and medication', 'Medication and rest',
       'Rest and fluids', 'Rest and fluids', 'Rest and fluids',
       'Medication and rest', 'Rest and fluids', 'Rest and fluids',
       'Medication and rest', 'Hospitalization and medication',
       'Rest and fluids', 'Rest and fluids',
       'Hospitalization and medication', 'Medication and rest',
       'Medication and rest', 'Rest and fluids', 'Rest and fluids',
       'Rest and fluids', 'Hospitalization and medication',
       'Rest and fluids', 'Rest and fluids', 'Medication and rest',
       'Rest and fluids', 'Hospitalization and medication',
       'Rest and fluids', 'Hospitalization and medication',
       'Medication and rest', 'Medication and rest', 'Rest and fluids',
       'Rest and fluids',

check test data classification report

In [10]:
print("Test classification Report : \n",classification_report(y_test,lr_test_pre))

Test classification Report : 
                                 precision    recall  f1-score   support

Hospitalization and medication       0.74      0.88      0.81        76
           Medication and rest       0.70      0.97      0.81        58
               Rest and fluids       0.99      0.86      0.92       266

                      accuracy                           0.88       400
                     macro avg       0.81      0.90      0.85       400
                  weighted avg       0.90      0.88      0.88       400



Check train data classification report

In [11]:
lr_train_pre = lr.predict(x_train_clean)
print("Train classification Report :  \n",classification_report(y_train,lr_train_pre))

Train classification Report :  
                                 precision    recall  f1-score   support

Hospitalization and medication       0.76      0.89      0.82       302
           Medication and rest       0.76      0.96      0.85       234
               Rest and fluids       0.99      0.88      0.93      1064

                      accuracy                           0.90      1600
                     macro avg       0.84      0.91      0.87      1600
                  weighted avg       0.91      0.90      0.90      1600



Check model predication using Confusion matrix

In [12]:
print(confusion_matrix(y_test,lr_test_pre))

[[ 67   7   2]
 [  2  56   0]
 [ 21  17 228]]


CatBoost Classifier

In [13]:
# define catboost classifier model
catboost = CatBoostClassifier(iterations=500,
learning_rate=0.05,
depth=6,
l2_leaf_reg = 3)

Fit model2 with train data

In [14]:
catboost.fit(x_train_clean,y_train)

0:	learn: 1.0394024	total: 172ms	remaining: 1m 25s
1:	learn: 0.9863332	total: 177ms	remaining: 44.2s
2:	learn: 0.9446803	total: 182ms	remaining: 30.2s
3:	learn: 0.9055000	total: 187ms	remaining: 23.2s
4:	learn: 0.8636609	total: 192ms	remaining: 19s
5:	learn: 0.8252830	total: 197ms	remaining: 16.2s
6:	learn: 0.7918301	total: 202ms	remaining: 14.2s
7:	learn: 0.7587559	total: 207ms	remaining: 12.7s
8:	learn: 0.7282656	total: 211ms	remaining: 11.5s
9:	learn: 0.7038599	total: 216ms	remaining: 10.6s
10:	learn: 0.6804804	total: 221ms	remaining: 9.83s
11:	learn: 0.6605176	total: 226ms	remaining: 9.2s
12:	learn: 0.6335809	total: 232ms	remaining: 8.69s
13:	learn: 0.6132108	total: 236ms	remaining: 8.21s
14:	learn: 0.5928841	total: 242ms	remaining: 7.82s
15:	learn: 0.5746357	total: 249ms	remaining: 7.53s
16:	learn: 0.5519768	total: 254ms	remaining: 7.23s
17:	learn: 0.5369234	total: 259ms	remaining: 6.94s
18:	learn: 0.5209335	total: 264ms	remaining: 6.67s
19:	learn: 0.5022712	total: 269ms	remaining

CatBoostClassifier(depth=6, iterations=500, l2_leaf_reg=3, learning_rate=0.05)

Predict output using model for both train and test data

In [15]:
ct_test_pred = catboost.predict(x_test_clean)
ct_train_pred = catboost.predict(x_train_clean)

Check model classification Report of Test data

In [16]:
print("Test classification Report : \n",classification_report(y_test,ct_test_pred))

Test classification Report : 
                                 precision    recall  f1-score   support

Hospitalization and medication       1.00      1.00      1.00        76
           Medication and rest       1.00      1.00      1.00        58
               Rest and fluids       1.00      1.00      1.00       266

                      accuracy                           1.00       400
                     macro avg       1.00      1.00      1.00       400
                  weighted avg       1.00      1.00      1.00       400



Check model prediction value using confusion Matrix Test data

In [17]:
print(confusion_matrix(y_test,ct_test_pred))

[[ 76   0   0]
 [  0  58   0]
 [  0   0 266]]


Check model classification report of train data

In [18]:
print("Train classification Report : \n",classification_report(y_train,ct_train_pred))

Train classification Report : 
                                 precision    recall  f1-score   support

Hospitalization and medication       1.00      1.00      1.00       302
           Medication and rest       1.00      1.00      1.00       234
               Rest and fluids       1.00      1.00      1.00      1064

                      accuracy                           1.00      1600
                     macro avg       1.00      1.00      1.00      1600
                  weighted avg       1.00      1.00      1.00      1600



Check model predictions using confusion matrix

In [19]:
print(confusion_matrix(y_train,ct_train_pred))

[[ 302    0    0]
 [   0  234    0]
 [   0    0 1064]]


Check model accuracy using cross val score

In [20]:
score = cross_val_score(catboost,x_train_clean,y_train,cv=5)
score.mean()

0:	learn: 1.0475036	total: 5.34ms	remaining: 2.66s
1:	learn: 0.9965495	total: 13.8ms	remaining: 3.44s
2:	learn: 0.9506183	total: 19.7ms	remaining: 3.27s
3:	learn: 0.9123722	total: 25.4ms	remaining: 3.15s


4:	learn: 0.8686940	total: 31.4ms	remaining: 3.1s
5:	learn: 0.8366753	total: 36.1ms	remaining: 2.97s
6:	learn: 0.8031779	total: 40.1ms	remaining: 2.82s
7:	learn: 0.7813355	total: 45.7ms	remaining: 2.81s
8:	learn: 0.7535930	total: 51.3ms	remaining: 2.8s
9:	learn: 0.7291412	total: 55.6ms	remaining: 2.72s
10:	learn: 0.7075468	total: 60.1ms	remaining: 2.67s
11:	learn: 0.6886231	total: 64.8ms	remaining: 2.63s
12:	learn: 0.6625399	total: 69.4ms	remaining: 2.6s
13:	learn: 0.6394374	total: 73.3ms	remaining: 2.54s
14:	learn: 0.6193134	total: 77.4ms	remaining: 2.5s
15:	learn: 0.6033877	total: 81.4ms	remaining: 2.46s
16:	learn: 0.5879950	total: 85.9ms	remaining: 2.44s
17:	learn: 0.5714424	total: 90.6ms	remaining: 2.43s
18:	learn: 0.5509039	total: 94.8ms	remaining: 2.4s
19:	learn: 0.5315418	total: 98.7ms	remaining: 2.37s
20:	learn: 0.5148617	total: 102ms	remaining: 2.34s
21:	learn: 0.4958774	total: 107ms	remaining: 2.32s
22:	learn: 0.4836437	total: 110ms	remaining: 2.29s
23:	learn: 0.4677252	total

np.float64(0.999375)

Select 2 model and make pipeline for data_cleaning and model2

In [21]:

full_pipeline = Pipeline([
    ("preprocessing",data_clean_pipeline),
    ("catboost",catboost)
])

Make File path name inside Model piplines folder

In [22]:
file_name = f"Disease_Treatment_Plan_ml_pipeline.pkl"
Pipeline_save_path = os.path.join(MODELS_PIPELINES,file_name)
Pipeline_save_path

'c:\\Users\\hamen\\OneDrive\\Desktop\\ml_project\\Diagnosis_ML_project\\Models_Pipelines\\Disease_Treatment_Plan_ml_pipeline.pkl'

Save ml model for Treatment_Plan

In [23]:
joblib.dump(full_pipeline,Pipeline_save_path)
print(f'save {Pipeline_save_path}')

save c:\Users\hamen\OneDrive\Desktop\ml_project\Diagnosis_ML_project\Models_Pipelines\Disease_Treatment_Plan_ml_pipeline.pkl
